# 🧫 EC2. Farmacoeconomía de la Terapia para la Superbacteria "Staph-X"

**Material desarrollado por:** Javier Morales, equipo [IA4LEGOS](https://ia4legos.umh.es/)

**Tema:** Generación de variables aleatorias.

**Licencia:** [CC BY-SA 4.0](http://creativecommons.org/licenses/by-sa/4.0/)

No olvides hacer una copia de este cuaderno (`Archivo > Guardar una copia en Drive`) antes de empezar a trabajar.

In [ ]:
#%%capture
# @title ⚠️ Cargar configuración del cuaderno
# Cargamos módulos de análisis numérico
import numpy as np          # importamos numpy como np
import pandas as pd         # importamos pandas como pd
import math                 # importamos módulo para cáculos matemáticos
import random
import inspect
from scipy import stats
import matplotlib.colors as mcolors # Importamos matplotlib.colors

# Cargamos módulos de análisis gráficos
from plotnine import *      # importamos módulo para gráficos con ggplot
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
%config InlineBackend.figure_format = 'retina'

#===============================================
# Cargar funciones bloque1
from urllib.request import urlretrieve
import re # for text manipulation


#url = 'https://raw.githubusercontent.com/asunmayoral/umh1477/refs/heads/main/bloque1_sps.py'
url = 'https://raw.githubusercontent.com/UMH1477/python/refs/heads/main/bloque1_sps.py'
urlretrieve(url, "bloque1_sps.py")
import bloque1_sps
from bloque1_sps import *

# visualización de las funciones precargadas
functions = []
for name, obj in inspect.getmembers(bloque1_sps):
    if inspect.isfunction(obj) and obj.__module__ == bloque1_sps.__name__:
        functions.append(name)

print("\nFunciones precargadas en bloque1_sps.py:\n")
for func_name in functions:
    print(f"- {func_name}")

# 1. Introducción

En el departamento de Bioestadística del **Hospital Central** se está evaluando el impacto económico de un nuevo tratamiento biológico frente a la infección por la superbacteria *Staph-X*. La eficacia del tratamiento depende del **fenotipo inmunológico** del paciente, lo que genera una respuesta heterogénea en la carga bacteriana y, por tanto, en los días de ingreso y en los costes de hospitalización.

Tu misión, como analista cuantitativo del hospital, es simular el escenario clínico para determinar el **coste medio por paciente** y el **riesgo de que el presupuesto hospitalario se desborde**.

Este estudio de caso está organizado en dos bloques:

* **Bloque Básico** modela el coste de un paciente individual en función de su fenotipo, sin tener en cuenta el contexto temporal (cuándo llega el paciente, ni si hay más pacientes a la vez).
* **Bloque Avanzado** sitúa ese mismo modelo dentro de un **brote epidémico de 90 días**, con tres fases de intensidad distinta que determinan tanto el volumen de pacientes como la mezcla de fenotipos, y añade una **restricción de capacidad** (número limitado de camas UCI) que obliga a externalizar pacientes a una clínica privada cuando se supera.

# <font color="brown">**1. Tu encargo**</font>

No se te pide solo código: se te pide una **recomendación fundamentada**. Al terminar cada bloque deberás entregar dos productos, como haría cualquier asesor de ciencia de datos en un proyecto real de consultoría:

* Un **informe técnico** que recoja el planteamiento del problema, el modelo utilizado, los resultados de la simulación (con sus intervalos de confianza — nunca un único número suelto) y tu interpretación de negocio de cada resultado.
* Una **presentación ejecutiva** pensada para la dirección del "Hospital Central": personas que no van a leer tu código ni tus fórmulas, y que necesitan entender, en el menor tiempo posible, cuál es el coste y el riesgo de la estrategia analizada, y qué recomiendas hacer al respecto.

Al final de cada bloque encontrarás el **encargo concreto de la Dirección**, redactado como una petición formal y organizado por objetivos — esa misma organización por objetivos es la que te conviene usar como esqueleto de tu informe técnico.

# 🅰️ <font color="brown">**2. Coste de la terapia por paciente**</font>

A continuación se describe la situación actual

## **2.1. El modelo de un paciente**

La población de pacientes se divide en tres grupos según su respuesta inmunológica latente ($T$):

| Fenotipo ($T$) | Descripción | Probabilidad ($p_j$) |
| :---: | :--- | :---: |
| **$T=1$** | Respondedor Rápido | $0.50$ |
| **$T=2$** | Respondedor Estándar | $0.40$ |
| **$T=3$** | Persistente (Resistente) | $0.10$ |

Para cada paciente, dos variables biológicas **independientes entre sí, condicionadas a $T$** determinan su evolución: la **Carga Bacteriana Inicial ($L$)** y la **Tasa de Depuración del Fármaco ($R$)**.

| Fenotipo ($T$) | Carga Inicial $L\mid T$ [UFC/ml] | Tasa de Depuración $R\mid T$ [log/día] |
| :--- | :---: | :---: |
| **1. Rápido** | $Ga(\alpha=2,\beta=1)$ | $U(1.5,\ 2.0)$ |
| **2. Estándar** | $Ga(\alpha=5,\beta=1)$ | $U(0.8,\ 1.2)$ |
| **3. Persistente** | $Ga(\alpha=10,\beta=1)$ | $U(0.2,\ 0.5)$ |

## **2.2. Función de coste**

El número de días que un paciente permanece ingresado ($D$) sigue una relación de eliminación logarítmica hasta un umbral de seguridad:

$$D = \max\left(0,\ \frac{\ln(L+1)}{R} + \epsilon\right), \qquad \epsilon \sim N(\mu=1,\ \sigma=0.2)$$

donde $\epsilon$ representa el ruido administrativo/logístico del hospital (independiente de $T$), y el $\max(0,\cdot)$ evita días de ingreso negativos.

El coste total por paciente ($Y$) incluye el fármaco, la estancia, y una penalización si la estancia se prolonga demasiado:

$$Y = C_{fijo} + (D\cdot C_{dia}) + \Psi(D), \qquad \Psi(D) = \begin{cases} 5\,000€ & \text{si } D>7 \\ 0 & \text{en otro caso} \end{cases}$$

| Parámetro | Valor | Significado |
| :--- | :---: | :--- |
| $C_{fijo}$ | $1\,200$ € | Coste del tratamiento biológico inicial |
| $C_{dia}$ | $600$ € | Coste de cama y cuidados por día |
| Umbral complicación | $7$ días | A partir de aquí se activa $\Psi(D)$ (coste UCI) |

## **2.3. Algoritmo de simulación**

1. Simular $t_i \sim T$ (fenotipos 1, 2, 3 con probabilidades 0.50, 0.40, 0.10).
2. Simular las variables condicionadas:
* si $t_i=1 \to l_i\sim Ga(2,1),\ r_i\sim U(1.5,2.0)$;
* si $t_i=2 \to l_i\sim Ga(5,1),\ r_i\sim U(0.8,1.2)$;
* si $t_i=3\to l_i\sim Ga(10,1),\ r_i\sim U(0.2,0.5)$.
3. Simular el ruido logístico $\epsilon_i \sim N(1, 0.2)$ (igual para los tres fenotipos).
4. Calcular los días de ingreso: $d_i = \max\left(0, \frac{\ln(l_i+1)}{r_i}+\epsilon_i\right)$.
5. Calcular el coste total: $y_i = 1200 + 600\, d_i + 5000\cdot\mathbb{I}_{\{d_i>7\}}$.
6. Repetir los pasos 1-5 $nsim$ veces para obtener la muestra $\{y_i\}$.

In [ ]:
# @title **Parámetros del modelo**

# --- Fenotipos y su distribución ---
FENOTIPOS = [1, 2, 3]              # 1=Rapido, 2=Estandar, 3=Persistente
PROBS_FENOTIPO = [0.50, 0.40, 0.10]

# --- Distribuciones condicionadas L|T, R|T ---
# L ~ Gamma(a=a_gamma, scale=1); R ~ Uniform(loc, scale)
PARAMS_FENOTIPO = {
    1: dict(a_gamma=2,  dist_R=("uniform", dict(loc=1.5, scale=0.5))),  # U(1.5, 2.0)
    2: dict(a_gamma=5,  dist_R=("uniform", dict(loc=0.8, scale=0.4))),  # U(0.8, 1.2)
    3: dict(a_gamma=10, dist_R=("uniform", dict(loc=0.2, scale=0.3))),  # U(0.2, 0.5)
}

# --- Parámetros económicos ---
C_FIJO = 1200.0    # € -- coste del tratamiento biologico inicial
C_DIA = 600.0      # €/dia -- coste de cama y cuidados
C_UCI = 5000.0     # € -- penalizacion por complicacion (estancia > 7 dias)
UMBRAL_D = 7.0     # dias

In [ ]:
# @title **Generador de pacientes**

def _generar_pacientes_base(T):
    """
    A partir de un vector de fenotipos T ya simulado, genera la Carga
    Inicial (L), la Tasa de Depuracion (R), el ruido logistico (epsilon)
    y los dias de ingreso (D), usando el metodo de composicion.

    Esta funcion es el "nucleo" reutilizado tanto por el simulador del
    Bloque Basico (simulador) como por el del Bloque Avanzado
    (simulador_brote): la fase del brote solo cambia CUANTOS pacientes
    hay de cada fenotipo, no la forma en que se generan sus variables
    biologicas una vez fijado el fenotipo.

    Returns
    -------
    L, R, eps, D : arrays de numpy, todos de longitud len(T)
    """
    n = len(T)
    L = np.zeros(n)
    R = np.zeros(n)
    for fen, cfg in PARAMS_FENOTIPO.items():
        mask = (T == fen)
        n_f = int(mask.sum())
        if n_f == 0:
            continue
        L[mask] = stats.gamma.rvs(a=cfg["a_gamma"], scale=1, size=n_f)
        dist_name, kwargs = cfg["dist_R"]
        R[mask] = getattr(stats, dist_name).rvs(size=n_f, **kwargs)

    eps = stats.norm.rvs(loc=1, scale=0.2, size=n)
    D = np.log(L + 1) / R + eps
    D = np.maximum(0, D)  # evita dias de ingreso negativos
    return L, R, eps, D

In [ ]:
# @title **Simulador**
def simulador(nsim):
    """
    Genera un banco de datos con `nsim` pacientes simulados, sin tener
    en cuenta el contexto temporal del brote (modelo base).

    Returns
    -------
    pd.DataFrame con columnas ['T','L','R','D','Complicacion','Y']
    """
    T = np.random.choice(FENOTIPOS, size=nsim, p=PROBS_FENOTIPO)
    L, R, eps, D = _generar_pacientes_base(T)
    complicacion = (D > UMBRAL_D).astype(int)
    Y = C_FIJO + C_DIA * D + C_UCI * complicacion
    return pd.DataFrame({"T": T, "L": L, "R": R, "D": D,
                          "Complicacion": complicacion, "Y": Y})

Comprobamos el simulador para conseguir $nsim=10.000$ pacientes simulados:

In [ ]:
NSIM = 10_000
datos_basico = simulador(NSIM)
datos_basico.head(10)

## **2.4. Verificar funcionamiento del algoritmo**

Antes de empezar a completar las tareas establecidas para analizar el comportamiento del sistema es necesario que verifiques las distribuciones asignadas en la descripción del proceso a partir de las 10000 simulaciones obtenidas.

Para dicha verificación puedes usar la función `gof_continuous` que te premite ajustar y estimar una distribución de tipo continuo  a un conjunto de datos. Para el análsiis de escenario climáticos basta con describir los resulttdos de dicha variable.

Te puedes ayudar tanto de resultados numéricos como de gráficos para verificar las distribuciones asumidas por la empresa y consideradas en el simulador. Presta especial atención a si la media y la varianza muestral de cada variable, calculadas por fenotipo, son coherentes con los parámetros teóricos.

## **2.5 El encargo de la Dirección**

> **MEMORÁNDUM INTERNO**
>
> **De:** Dirección del "Hospital Central"
> **Para:** Equipo de Ciencia de Datos / Bioestadística
> **Asunto:** Diagnóstico del coste y el riesgo financiero del tratamiento de Staph-X
>
> Antes de decidir si generalizamos este tratamiento, necesitamos entender con números —no con impresiones— su coste real y el riesgo de desviación presupuestaria. Trabajad con la muestra de 10.000 pacientes que habéis generado (`datos_basico`) y acompañad **cada estimación de su intervalo de confianza al 95%**: una cifra sin margen de error no nos sirve para tomar una decisión de este calado. Os hemos organizado la petición en tres objetivos.

### Objetivo 1. Cuantificar el riesgo económico de un paciente

Necesitamos saber, en cifras concretas, qué podemos esperar de un paciente cualquiera y hasta qué punto se pueden torcer las cosas en el peor de los casos.

* **O1.1.** Estimad el **coste esperado** $E(Y)$ y la **probabilidad de complicación** $Pr(D>7)$.
* **O1.2.** Calculad el **percentil 95** de $Y$, mediante macro-réplicas. ¿Qué significado tiene este valor para la provisión de fondos del hospital, frente a $E(Y)$?
* **O1.3.** Estimad el coeficiente de variación de $Y$ para cada fenotipo por separado. ¿Qué fenotipo presenta mayor incertidumbre relativa en el coste, y qué implicación tiene eso para la gestión del riesgo financiero del hospital?

### Objetivo 2. Entender qué hay detrás de esa variabilidad

Antes de proponer ninguna solución, queremos saber qué parte del proceso explica realmente el riesgo.

* **O2.1.** Descomponed, por simulación, la varianza total de $Y$ en la parte atribuible al término lineal ($600\cdot D$) y la parte atribuible al salto discreto ($5000\cdot\mathbb{I}_{\{D>7\}}$). ¿Cuál domina la variabilidad del coste?
* **O2.2.** Diseñad una prueba de sensibilidad global que determine cuál de los parámetros del modelo (probabilidades de fenotipo, parámetros de la Gamma, de la Uniforme, o el ruido $\epsilon$) explica mayor proporción de la varianza de $Y$: necesitamos saber dónde concentrar nuestros esfuerzos.

### Objetivo 3. Explorar palancas de mejora del tratamiento actual

Antes de cambiar de estrategia de raíz, queremos agotar los ajustes más baratos sobre el tratamiento que ya tenemos.

* **O3.1.** Un fabricante nos ofrece un tratamiento genérico más barato ($C_{fijo}=600$€) pero menos eficaz (mayor carga bacteriana esperada). Planteadnos cómo compararíais, por simulación, este tratamiento frente al actual, en términos de coste esperado y de riesgo (percentil 95).
* **O3.2.** Investigad qué es el **ICER** (Incremental Cost-Effectiveness Ratio) en farmacoeconomía y plantead cómo lo calcularíais si un nuevo fármaco redujera a la mitad la tasa inversa de depuración del fenotipo Persistente (aumentando $R$), a un coste adicional de 800€ por tratamiento.
* **O3.3.** Estamos negociando con el fabricante un contrato de "pago por resultados": nos devolvería el 50% del $C_{fijo}$ si el paciente desarrolla complicación. Rediseñad $Y$ bajo este contrato y estimad, por simulación, el ahorro esperado para el hospital.
* **O3.4.** Diseñad un experimento de simulación para estudiar cómo cambiaría la distribución de $Y$ si el umbral de complicación (7 días) se recalibrara a 5 o a 10 días. ¿Qué implicación clínica y económica tendría cada cambio?

# 🅱️ 3. Gestión de la capacidad UCI durante un brote

El Bloque Básico trata a cada paciente como un caso aislado. En un **brote real**, los pacientes no llegan de uno en uno de forma homogénea en el tiempo: llegan en oleadas, y la propia intensidad del brote afecta tanto al **volumen de ingresos** como a la **mezcla de fenotipos** (la presión selectiva de un uso masivo de antibióticos durante el pico favorece la aparición de cepas más resistentes). Además, el hospital dispone de un **número limitado de camas UCI** reservadas para esta patología: cuando la demanda supera esa capacidad en un día concreto, los pacientes excedentes deben derivarse a una clínica privada, a un coste mayor.

## 3.1. El modelo temporal del brote y la restricción de capacidad

El brote se desarrolla a lo largo de $ndias=90$ días, divididos en tres fases. El número de pacientes que ingresan cada día se modela como un **proceso de Poisson cuya tasa depende de la fase** (Poisson por tramos):

$$N_d \sim Poisson(\lambda_{f(d)}), \qquad d=1,\dots,90$$

**Tabla 1. Fases del brote, duración y tasa de ingreso de pacientes**

| Fase | Días | Duración (días) | Intensidad del brote | $\lambda$ (pacientes/día) |
| :--- | :---: | :---: | :---: | :---: |
| **Inicio** | 1–20 | 20 | Baja | $6$ |
| **Pico** | 21–50 | 30 | Alta | $25$ |
| **Remisión** | 51–90 | 40 | Media-baja | $10$ |

**Tabla 2. Mezcla de fenotipos por fase** (presión selectiva del antibiótico: durante el Pico aumenta la proporción de fenotipo Persistente)

| Fase | $p(T=1)$ Rápido______ | $p(T=2)$ Estándar______ | $p(T=3)$ Persistente______ |
| :--- | :---: | :---: | :---: |
| **Inicio** | $0.65$ | $0.30$ | $0.05$ |
| **Pico** | $0.45$ | $0.35$ | $0.20$ |
| **Remisión** | $0.58$ | $0.32$ | $0.10$ |

Una vez fijados el día (fase) y el fenotipo $T$ de un paciente, sus variables biológicas $L$, $R$, $\epsilon$ y sus días de ingreso $D$ se generan **exactamente igual que en el Bloque Básico**: la fase del brote solo determina cuántos pacientes hay y de qué fenotipo, no cómo se comporta cada paciente individual una vez ingresado.

El hospital dispone de $B_{UCI}=3$ camas UCI reservadas para *Staph-X*. Cada día, de todos los pacientes que ingresan ESE día y que acaban desarrollando complicación ($D>7$), como mucho $B_{UCI}$ pueden ser atendidos con cama UCI propia (coste $C_{UCI}=5\,000$€); el resto debe derivarse a una clínica privada externa, a un coste mayor $C_{UCI}^{ext}=9\,000$€. Qué pacientes concretos consiguen la cama se determina por **orden de llegada aleatorio dentro del día** (no por fenotipo ni por gravedad).

Para cada paciente $i$ del día $d$ (fase $f(d)$) se calculan **dos costes** a partir de las MISMAS variables base $L_i, R_i, \epsilon_i$ (y por tanto la misma $D_i$):

$$
\begin{aligned}
Y_i^{\,sin} &= C_{fijo} + C_{dia}\, D_i + C_{UCI}\cdot \mathbb{I}_{\{D_i>7\}} &\text{(coste ignorando la restricción de capacidad)}\\[4pt]
Y_i^{\,con} &= C_{fijo} + C_{dia}\, D_i + \begin{cases} 0 & D_i\le 7\\ C_{UCI} & D_i>7 \text{ y consigue cama}\\ C_{UCI}^{ext} & D_i>7 \text{ y no consigue cama}\end{cases} &\text{(coste real, con capacidad limitada)}
\end{aligned}
$$

El **sobrecoste atribuible a la falta de capacidad** es $\Delta_i = Y_i^{con} - Y_i^{sin}$.

> 💡 **Nota metodológica — simulación pareada.** $Y_i^{sin}$ e $Y_i^{con}$ comparten exactamente las mismas variables base: solo cambia el coste que se asigna cuando hay complicación. Esta técnica de **números aleatorios comunes** permite estimar $\Delta_i$ con mucha más precisión que si se simularan ambos escenarios de forma independiente.

| Parámetro | Símbolo______ | Valor______ |
| :--- | :--- | :--- |
| Camas UCI reservadas | $B_{UCI}$ | $3$ |
| Coste UCI propia | $C_{UCI}$ | $5\,000$ € |
| Coste clínica externa | $C_{UCI}^{ext}$ | $9\,000$ € |
| Horizonte del brote | $ndias$ | $90$ días |

## 3.2. Algoritmo de simulación extendido

1. Para cada día $d=1,\dots,90$, determinar su fase $f(d)$ (Tabla 1) y simular $n_d\sim Poisson(\lambda_{f(d)})$ pacientes.
2. Para cada uno de los $n_d$ pacientes, simular su fenotipo $T_i$ según la mezcla de la fase (Tabla 2), y a partir de él, $L_i, R_i, \epsilon_i, D_i$ (igual que en el Bloque Básico).
3. Calcular $Y_i^{sin}$ (sin restricción de capacidad).
4. Dentro de cada día, asignar un orden de llegada aleatorio a los pacientes complicados ($D_i>7$); los $B_{UCI}$ primeros consiguen cama UCI propia, el resto se derivan a la clínica externa.
5. Calcular $Y_i^{con}$ y $\Delta_i = Y_i^{con}-Y_i^{sin}$ según corresponda.
6. Repetir los pasos 1-5 para los 90 días, obteniendo el registro completo del brote.

In [ ]:
# @title Parámetros del modelo temporal (Bloque Avanzado)

# --- Fases del brote: (nombre, dia_inicio, dia_fin, lambda pacientes/dia) ---
FASES_DIAS = [
    ("Inicio",    1, 20, 6),
    ("Pico",     21, 50, 25),
    ("Remision", 51, 90, 10),
]
NDIAS_BROTE = 90

# --- Mezcla de fenotipos por fase (Tabla 2) ---
PROBS_FENOTIPO_FASE = {
    "Inicio":   [0.65, 0.30, 0.05],
    "Pico":     [0.45, 0.35, 0.20],
    "Remision": [0.58, 0.32, 0.10],
}

# --- Restricción de capacidad UCI ---
B_UCI = 3           # camas UCI reservadas para Staph-X
C_UCI_EXT = 9000.0  # € -- coste de derivar un paciente a clinica privada

def _fase_de_dia(dia):
    for nombre, d0, d1, lam in FASES_DIAS:
        if d0 <= dia <= d1:
            return nombre, lam
    raise ValueError(f"dia {dia} fuera de rango")

In [ ]:
# @title Generador de datos: simulador_brote(ndias)

def simulador_brote(ndias=NDIAS_BROTE):
    """
    Simula un brote completo de `ndias` dias, con tres fases de
    intensidad distinta (Tabla 1), mezcla de fenotipos dependiente de
    la fase (Tabla 2), y una restriccion de capacidad de B_UCI camas.

    Returns
    -------
    pd.DataFrame con un registro por paciente y columnas:
        ['Dia','Fase','T','L','R','D','Complicacion','Orden',
         'Y_sin_capacidad','Uso_UCI','Y_con_capacidad','Sobrecoste_capacidad']
    """
    dias = np.arange(1, ndias + 1)
    fase_dia = np.array([_fase_de_dia(d)[0] for d in dias])
    lam_dia = np.array([_fase_de_dia(d)[1] for d in dias])

    # 1. Numero de pacientes que ingresan cada dia
    n_pac_dia = np.random.poisson(lam_dia)
    total_n = int(n_pac_dia.sum())

    Dia = np.repeat(dias, n_pac_dia)
    Fase = np.repeat(fase_dia, n_pac_dia)

    # 2. Fenotipo de cada paciente, segun la mezcla de SU fase
    T = np.zeros(total_n, dtype=int)
    for fase in PROBS_FENOTIPO_FASE:
        mask = (Fase == fase)
        n_f = int(mask.sum())
        if n_f == 0:
            continue
        T[mask] = np.random.choice(FENOTIPOS, size=n_f, p=PROBS_FENOTIPO_FASE[fase])

    # Variables biologicas: mismo nucleo que el Bloque Basico
    L, R, eps, D = _generar_pacientes_base(T)
    complicacion = (D > UMBRAL_D).astype(int)

    # 3. Coste SIN restriccion de capacidad
    Y_sin_capacidad = C_FIJO + C_DIA * D + C_UCI * complicacion

    # 4. Orden de llegada aleatorio dentro del dia (independiente del fenotipo)
    orden = np.random.uniform(size=total_n)

    datos = pd.DataFrame({
        "Dia": Dia, "Fase": Fase, "T": T, "L": L, "R": R, "D": D,
        "Complicacion": complicacion, "Orden": orden,
        "Y_sin_capacidad": Y_sin_capacidad,
    })

    # Entre los pacientes complicados de un mismo dia, los B_UCI con menor
    # "Orden" consiguen cama propia; el resto van a la clinica externa.
    datos["rank_dia"] = np.inf
    comp_mask = datos["Complicacion"] == 1
    datos.loc[comp_mask, "rank_dia"] = (
        datos.loc[comp_mask].groupby("Dia")["Orden"].rank(method="first")
    )
    datos["Uso_UCI"] = np.where(
        ~comp_mask, "No_aplica",
        np.where(datos["rank_dia"] <= B_UCI, "UCI_estandar", "Clinica_externa")
    )

    # 5. Coste CON restriccion de capacidad
    coste_complicacion = np.select(
        [datos["Uso_UCI"] == "UCI_estandar", datos["Uso_UCI"] == "Clinica_externa"],
        [C_UCI, C_UCI_EXT], default=0.0
    )
    datos["Y_con_capacidad"] = C_FIJO + C_DIA * datos["D"] + coste_complicacion
    datos["Sobrecoste_capacidad"] = datos["Y_con_capacidad"] - datos["Y_sin_capacidad"]

    return datos.drop(columns="rank_dia")

Podemos simular el brote completo con:

In [ ]:
datos_brote = simulador_brote(NDIAS_BROTE)
datos_brote.head(10)

## **3.3. Verificar funcionamiento del algoritmo**

Antes de empezar a completar las tareas establecidas para analizar el comportamiento del sistema es necesario que verifiques las distribuciones asignadas en la descripción del proceso a partir de las simulaciones obtenidas.

Para dicha verificación puedes usar la función `gof_continuous` que te premite ajustar y estimar una distribución de tipo continuo a un conjunto de datos. Para el análisis del número de pacientes por día y de la mezcla de fenotipos basta con describir los resultados de dichas variables.

Te puedes ayudar tanto de resultados numéricos como de gráficos para verificar las distribuciones asumidas por el hospital y consideradas en el simulador. Presta especial atención a si la media y la varianza muestral de cada variable, calculadas por fase, son coherentes con los parámetros teóricos.

Obtén, con su IC al 95%, mediante macro-réplicas del brote completo:

* El **coste total esperado** del brote.
* El **sobrecoste total esperado** por falta de capacidad (y qué % supone).
* El **número esperado de días con desbordamiento** y de **pacientes externalizados**.
* El **percentil 95** del coste total del brote.

## **3.4 El encargo de la Dirección (fase 2): ¿compensa ampliar la capacidad UCI?**

> **MEMORÁNDUM INTERNO**
>
> **De:** Dirección del "Hospital Central"
> **Para:** Equipo de Ciencia de Datos / Bioestadística
> **Asunto:** Evaluación del impacto de la restricción de capacidad UCI durante un brote
>
> Gracias por el diagnóstico de la fase anterior. Ahora necesitamos que evaluéis cuánto nos cuesta la limitación actual de camas UCI durante un brote, y qué podemos hacer al respecto. Trabajad siempre sobre los registros pareados de `datos_brote` (`Y_sin_capacidad` / `Y_con_capacidad`), generados con números aleatorios comunes, y acompañad cada estimación de su intervalo de confianza al 95%. Os hemos organizado la petición en cuatro objetivos, más una ampliación optativa.


### Objetivo 4. Comparar, brote a brote, el coste con y sin restricción de capacidad

* **O4.1.** Estimad el **coste total esperado** del brote con y sin restricción de capacidad, $E(\text{Coste\_total\_con})$ y $E(\text{Coste\_total\_sin})$, y el **sobrecoste medio** atribuible a la falta de capacidad, $E(\text{Sobrecoste\_total})$. Aprovechad que el diseño es pareado (números aleatorios comunes) para explicarnos por qué el intervalo de confianza del sobrecoste sale mucho más estrecho que si hubierais simulado ambos escenarios de forma independiente.
* **O4.2.** Calculad el **número esperado de días con desbordamiento** y el **número esperado de pacientes externalizados** a la clínica privada.
* **O4.3.** Expresad el sobrecoste total esperado como porcentaje del coste total del brote. ¿Qué magnitud tiene ese porcentaje, y os parece asumible?

### Objetivo 5. Saber de qué depende ese sobrecoste

* **O5.1.** Investigad el efecto de una política de aislamiento preventivo que redujera $\lambda_{Pico}$ de 25 a 18 pacientes/día. Cuantificad, por simulación, el ahorro esperado en sobrecoste frente al coste de implementar esas medidas.
* **O5.2.** La externalización cuesta $C_{UCI}^{ext}=9\,000$€ fijos por paciente. Cuantificad cómo cambiaría el sobrecoste total esperado si, en lugar de ese coste fijo, se negociara con la clínica privada un contrato de coste variable decreciente según el volumen de pacientes derivados.
* **O5.3.** Descomponed, por simulación, el sobrecoste medio por falta de capacidad según la fase del brote en la que se produce. ¿En qué fase resulta más gravosa la restricción de capacidad? Relacionadlo con el volumen de pacientes y la mezcla de fenotipos de cada fase (Tablas 1 y 2).

### Objetivo 6. Dimensionar la capacidad UCI y comprobar su robustez

* **O6.1.** Calculad, por simulación, la relación entre el número de camas UCI reservadas ($B_{UCI}=1,2,3,4,5$) y el sobrecoste total esperado del brote. ¿A partir de qué número de camas el ahorro marginal de añadir una cama más deja de compensar (asumiendo un coste de oportunidad por cama reservada)?
* **O6.2.** Diseñad un procedimiento de simulación para estimar la probabilidad de que se agote la capacidad UCI en dos días consecutivos.
* **O6.3.** Comparad, por simulación: (a) 5 camas UCI fijas todo el año, frente a (b) 3 camas fijas más la opción de alquilar hasta 4 adicionales solo cuando se necesiten (a mayor coste por uso). Plantead los supuestos de coste necesarios para comparar ambas opciones.

### Objetivo 7. Vuestra recomendación

* **O7.1.** Redactad, a partir de los resultados de los Objetivos 4 a 6, las tres recomendaciones principales que le haríais a la dirección del hospital de cara al PRÓXIMO brote de Staph-X, justificando cada una con una métrica concreta calculable con este estudio de caso.

## Objetivo 8. Ampliación optativa

Lo que sigue **no forma parte del encargo formal** de la Dirección: es un banco de cuestiones adicionales, de mayor dificultad, para quien quiera explorar el modelo del brote con más profundidad.

* **O8.1.** Demostrad por qué el diseño pareado ($Y^{sin}$ e $Y^{con}$ calculados a partir de las mismas $D,L,R,\epsilon$) reduce la varianza de la estimación del sobrecoste por falta de capacidad, frente a simular ambos escenarios de forma independiente. Verificadlo empíricamente.
* **O8.2.** Comparad este modelo de capacidad (comprobación diaria, independiente entre días) con un modelo de colas $M/M/c/K$, donde un paciente complicado podría ocupar una cama varios días consecutivos. ¿Qué sesgo introduce la simplificación "un día, una decisión" del modelo actual?
* **O8.3.** Formulad como un problema de optimización bajo incertidumbre la decisión conjunta de cuántas camas UCI reservar y cuánto invertir en diagnóstico precoz, dado un presupuesto total limitado, usando los resultados de vuestras simulaciones como función objetivo.
* **O8.4.** Investigad qué es un **proceso de Cox** (Poisson doblemente estocástico) y plantead cómo mejoraríais el modelo si, además de la fase, la propia intensidad del brote (número reproductivo efectivo) fuera incierta día a día.
* **O8.5.** El hospital podría alquilar camas UCI adicionales solo durante el Pico (días 21-50), pagando una cuota diaria fija por cama disponible. Diseñad, por simulación, cómo determinaríais la cuota máxima que el hospital debería estar dispuesto a pagar.
* **O8.6.** El hospital podría invertir en diagnóstico temprano, reduciendo la carga bacteriana inicial esperada del fenotipo Persistente. Diseñad un análisis coste-beneficio, por simulación, que compare el coste de ese programa frente al ahorro esperado en sobrecoste de capacidad.